# 02 — Player analysis

Works off the DuckDB warehouse built by `fpl build`. If the cell below says the warehouse doesn't exist, run `fpl run` from a terminal in the repo first.

The point of this notebook: **write SQL against the marts**, exactly as you would in Snowflake, and only drop into pandas for charts.

In [ ]:
# Run this cell first in every notebook.
# It makes the package importable from the repo root and loads settings from config/settings.toml.
# autoreload: edits under src/ take effect on the next cell run, no kernel restart needed.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from fpl_analysis.config import load_settings
from fpl_analysis.store import connect, query

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
settings = load_settings()
print("project root:", settings.project_root)
print("warehouse   :", settings.duckdb_path, "| exists:", settings.duckdb_path.exists())

In [ ]:
# What's in the warehouse?
query(settings, """
SELECT table_schema, table_name, table_type
FROM information_schema.tables
WHERE table_schema IN ('raw', 'history', 'staging', 'marts')
ORDER BY 1, 2
""")

## Snapshot context

In [ ]:
query(settings, "SELECT * FROM staging.stg_snapshot")

## Top players by modelled expected points over the horizon

In [ ]:
query(settings, """
SELECT rank_overall, web_name, team_short_name, position_code, price_m, selected_by_pct,
       form_signal, season_ppg, xgi_per_90, availability, fixture_run, ep_next, ep_horizon, ep_horizon_per_m
FROM marts.mart_player_horizon
WHERE availability > 0
ORDER BY ep_horizon DESC
LIMIT 25
""")

## Model vs the API's own `ep_next`

A useful sanity check: where the model and FPL's own expected-points figure disagree most, look at *why* (fixture, minutes, form quirks).

In [ ]:
cmp = query(settings, """
SELECT web_name, team_short_name, position_code, next_fixture_label, form_signal, availability,
       ep_next AS model_ep_next, api_ep_next, ep_next - api_ep_next AS delta
FROM marts.mart_player_horizon
WHERE api_ep_next IS NOT NULL AND fixtures_next_gw > 0 AND minutes >= 180
ORDER BY ABS(ep_next - api_ep_next) DESC
LIMIT 20
""")
cmp

In [ ]:
import matplotlib.pyplot as plt

df = query(settings, """
SELECT ep_next, api_ep_next, position_code
FROM marts.mart_player_horizon
WHERE api_ep_next IS NOT NULL AND fixtures_next_gw > 0 AND minutes >= 180
""")
fig, ax = plt.subplots(figsize=(6, 6))
for pos, grp in df.groupby("position_code"):
    ax.scatter(grp["api_ep_next"], grp["ep_next"], label=pos, alpha=0.6, s=18)
lim = max(df["ep_next"].max(), df["api_ep_next"].max()) + 0.5
ax.plot([0, lim], [0, lim], color="grey", linewidth=1)
ax.set_xlabel("FPL API ep_next")
ax.set_ylabel("Model ep_next")
ax.set_title("Model vs API expected points, next GW")
ax.legend()
plt.show()

## Fixture swing: who has the best run over the horizon?

In [ ]:
query(settings, """
SELECT fixture_rank, team_short_name, fixtures_in_horizon, home_fixtures, ROUND(avg_fdr, 2) AS avg_fdr,
       ROUND(fixture_score, 2) AS fixture_score, fixture_run
FROM marts.mart_team_fixture_summary
ORDER BY fixture_rank
""")

## Price and ownership over time (needs 2+ snapshots)

`history.player_snapshot` accumulates one row per player per refresh. Refresh daily around the price-change window (~02:30 UK) and this becomes your price-tracker.

In [ ]:
hist = query(settings, """
SELECT h.snapshot_ts, h.player_id, p.web_name, h.now_cost / 10.0 AS price_m, h.selected_by_percent, h.form
FROM history.player_snapshot AS h
JOIN staging.stg_players AS p ON p.player_id = h.player_id
WHERE h.player_id IN (SELECT player_id FROM marts.mart_player_horizon ORDER BY ep_horizon DESC LIMIT 5)
ORDER BY h.player_id, h.snapshot_ts
""")
print(f"{hist['snapshot_ts'].nunique()} snapshot(s) in history")
hist.pivot_table(index="snapshot_ts", columns="web_name", values="price_m")

## Scratch SQL

Use this cell for ad-hoc questions. The `raw` schema has the untouched API columns if you need something staging doesn't expose yet — and if you find yourself reusing a query, promote it to a model in `sql/`.

In [ ]:
query(settings, """
SELECT position_code, COUNT(*) AS players, ROUND(AVG(price_m), 2) AS avg_price, ROUND(AVG(ep_horizon), 2) AS avg_ep
FROM marts.mart_player_horizon
GROUP BY position_code
ORDER BY position_code
""")